# ORIZZONTE — Ingestion corpus con MinerU (GPU)

Parsa tutti i PDF del corpus e produce **JSON nello stesso formato di `parsing.py`** (`text_markdown`, `topic_id`, `difficulty_levels`, ...), pronti da scompattare direttamente in `data/processed/parsed/` e dare in pasto a `chuncking.py`.

Testato con **MinerU 3.4.4**, backend `pipeline` (l'11/07/2026, promosso su 3 PDF campione vs Docling: gerarchia titoli pari o migliore, formule inline in più, OCR nettamente superiore sulle scansioni).

**Setup richiesto:**
1. Carica la cartella `data/raw` (con le sottocartelle TOPIC) come dataset Kaggle, es. `orizzonte-corpus-raw`
2. Aggiungi il dataset a questo notebook (Add Input)
3. Settings → Accelerator: **GPU T4 x2** (ne basta una) · Internet: **ON**
4. Esegui tutto; alla fine scarica `corpus_processed.zip` dall'output e scompattalo in `data/processed/parsed/` (sovrascrive i JSON Docling omonimi; i file `api_*` non vengono toccati)
5. In locale: cancella `data/processed/chunks/` e `data/vector_db/`, poi rilancia `chuncking.py` e `indexing.py`

In [ ]:
# ── 1. Verifica GPU ──
import torch
print("CUDA disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
# ── 2. Installazione MinerU (backend pipeline, adatto alla T4) + PyMuPDF per il conteggio pagine ──
%pip install -q "mineru[core]" PyMuPDF
!mineru --version

In [ ]:
# ── 3. Configurazione percorsi ──
from pathlib import Path

# ADATTA QUESTO allo slug del tuo dataset:
RAW_DIR = Path("/kaggle/input/orizzonte-corpus-raw")
OUT_DIR = Path("/kaggle/working/processed")
TMP_DIR = Path("/kaggle/working/_mineru_tmp")
OUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(RAW_DIR.rglob("*.pdf"))
print(f"{len(pdfs)} PDF trovati")
for p in pdfs[:5]:
    print("  ", p.relative_to(RAW_DIR))

In [ ]:
# ── 4. Ingestion: MinerU su ogni PDF → JSON nel formato di parsing.py ──
import json, re, shutil, subprocess, time
from datetime import datetime
import fitz  # PyMuPDF

LEVEL_PREFIX_RE = re.compile(r"^([ABCD](?:\s*,\s*[ABCD])*)\s*-\s*")
TOPIC_RE = re.compile(r"^TOPIC\s+(\d+)\s*-\s*(.+)$")
LEVEL_MAP = {"A": "bambini", "B": "medie", "C": "superiori", "D": "universitari"}

def extract_metadata(pdf: Path) -> dict:
    """Metadati nello stesso schema dei JSON prodotti da src/parsing.py."""
    folder = pdf.parent.name if pdf.parent != RAW_DIR else "SENZA_TOPIC"
    tm = TOPIC_RE.match(folder)
    lm = LEVEL_PREFIX_RE.match(pdf.stem)
    levels = [x.strip() for x in lm.group(1).split(",")] if lm else ["unknown"]
    return {
        "source_file": f"data/raw/{folder}/{pdf.name}",
        "topic_id": int(tm.group(1)) if tm else 0,
        "topic_name": tm.group(2).strip() if tm else folder,
        "difficulty_levels": levels,
        "difficulty_labels": [LEVEL_MAP.get(l, "sconosciuto") for l in levels],
        "title": LEVEL_PREFIX_RE.sub("", pdf.stem).strip(),
    }

def parse_with_mineru(pdf: Path) -> str:
    out = TMP_DIR / pdf.stem
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    # '-b pipeline' esplicito: il default di MinerU 3.x (hybrid-engine) è troppo
    # pesante per la T4; pipeline è il backend validato nel test dell'11/07/2026
    proc = subprocess.run(["mineru", "-p", str(pdf), "-o", str(out), "-b", "pipeline"],
                          capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"mineru exit {proc.returncode}: {proc.stderr.strip()[-300:]}")
    md_files = sorted(out.rglob("*.md"))
    if not md_files:
        raise RuntimeError("nessun .md prodotto")
    best = max(md_files, key=lambda f: f.stat().st_size)
    return best.read_text(encoding="utf-8")

manifest, t0 = [], time.time()
for i, pdf in enumerate(pdfs, 1):
    meta = extract_metadata(pdf)
    rel_dir = pdf.parent.relative_to(RAW_DIR)
    dest = OUT_DIR / rel_dir
    dest.mkdir(parents=True, exist_ok=True)
    json_path = dest / (pdf.stem + ".json")

    if json_path.exists():
        print(f"[{i:>2}/{len(pdfs)}] ↷ già fatto: {pdf.name}")
        manifest.append({**meta, "status": "skipped"})
        continue

    print(f"[{i:>2}/{len(pdfs)}] ⚙ {pdf.name}", flush=True)
    t = time.time()
    try:
        md = parse_with_mineru(pdf)
        with fitz.open(str(pdf)) as doc:
            num_pages = len(doc)
        n_headers = len(re.findall(r"^#{1,6}\s", md, flags=re.M))
        result = {**meta, "num_pages": num_pages, "text_markdown": md}
        json_path.write_text(json.dumps(result, ensure_ascii=False, indent=2),
                             encoding="utf-8")
        print(f"      ✓ {len(md)//1000} kchar, {n_headers} header, {num_pages} pag — {time.time()-t:.0f}s")
        manifest.append({**meta, "status": "ok", "chars": len(md), "headers": n_headers,
                         "processed_at": datetime.now().isoformat(timespec="seconds")})
    except Exception as e:
        print(f"      ✗ {type(e).__name__}: {e}")
        manifest.append({**meta, "status": f"failed: {type(e).__name__}"})

# Manifest FUORI da OUT_DIR: se finisse nello zip, chuncking.py proverebbe a chunkarlo
Path("/kaggle/working/manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
ok = sum(1 for m in manifest if m["status"] == "ok")
print(f"\nFinito in {(time.time()-t0)/60:.0f} min — ok: {ok}, "
      f"falliti: {sum(1 for m in manifest if m['status'].startswith('failed'))}")

In [ ]:
# ── 5. Zip dei risultati (JSON in formato parsing.py) ──
import shutil
shutil.make_archive("/kaggle/working/corpus_processed", "zip", OUT_DIR)
!ls -lh /kaggle/working/corpus_processed.zip
print("""
Scarica corpus_processed.zip dal pannello Output, poi in locale:
  1. Scompattalo DENTRO data/processed/parsed/ (merge: sovrascrive i JSON
     Docling omonimi, lascia intatti i file api_* di Vikidia/Wikipedia/EduINAF)
  2. Remove-Item -Recurse -Force data\\processed\\chunks
  3. Remove-Item -Recurse -Force data\\vector_db
  4. python src/chuncking.py
  5. python src/indexing.py""")